# Notebook 02 - Fine-Tuning de LLM para Dominio Medico

## Tech Challenge Fase 3 - Assistente Virtual Medico

Este notebook cobre:
1. Configuracao do pipeline de fine-tuning
2. Preparacao dos dados para treinamento
3. Fine-tuning com LoRA/QLoRA (eficiente em memoria)
4. Avaliacao do modelo treinado
5. Exportacao do modelo

---
## 1. Conceitos de Fine-Tuning

O fine-tuning consiste em adaptar um modelo LLM generico (LLaMA, Falcon, etc.) a um dominio especifico, utilizando dados proprios do hospital.

### Por que Fine-Tuning?
- O modelo generico nao conhece protocolos especificos do hospital
- Fine-tuning permite personalizar o modelo para o contexto medico
- Resultados mais precisos e consistentes em relacao ao dominio

### Tecnicas Utilizadas:
- **LoRA (Low-Rank Adaptation)**: Ajusta apenas pesos de baixa rank, economizando memoria
- **QLoRA**: LoRA com quantizacao 4-bit, permitindo treinamento em GPUs menores
- **Prompt Tuning**: Ajusta apenas os embeddings do prompt

---
## 2. Instalacao e Configuracao

In [ ]:
# Instalacao de dependencias para fine-tuning
!pip install transformers datasets accelerate peft bitsandbytes trl -q

In [ ]:
import os
import json
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer
from dotenv import load_dotenv

load_dotenv()

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

---
## 3. Carregamento do Dataset

In [ ]:
# Carregar dados preparados (dataset com618 exemplos)
dataset_file = "../data/dataset_alpaca.json"

with open(dataset_file, 'r', encoding='utf-8') as f:
    dados_treino = json.load(f)

# Garantir que todos os registros tenham o campo 'text'
for item in dados_treino:
    if 'text' not in item:
        instruction = item.get('instruction', '')
        inp = item.get('input', '')
        output = item.get('output', '')
        item['text'] = f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n{output}"

print(f"Total de exemplos: {len(dados_treino)}")
print(f"\nFontes dos dados:")

# Contar por fonte
fontes = {}
for item in dados_treino:
    fonte = item.get('source', 'desconhecida')
    fontes[fonte] = fontes.get(fonte, 0) + 1

for fonte, count in fontes.items():
    print(f"  - {fonte}: {count} exemplos")

print(f"\nExemplo formatado:")
print(dados_treino[0]['text'][:500])

# Converter para HuggingFace Dataset
dataset = Dataset.from_list(dados_treino)
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f"\nDataset de treino: {len(dataset['train'])} exemplos")
print(f"Dataset de validacao: {len(dataset['test'])} exemplos")

---
## 4. Configuracao do Modelo Base

### Escolha do Modelo

Para este projeto, utilizamos:
- **TinyLlama/TinyLlama-1.1B-Chat-v1.0** - Modelo leve e eficiente para fine-tuning
- Alternativas: Mistral-7B, LLaMA-2-7B, Falcon-7B

### Quantizacao 4-bit (QLoRA)
Reduz o consumo de memoria permitindo treinamento em GPUs com menor VRAM.

In [ ]:
# Modelo base - TinyLlama para demo (trocar por modelo maior em producao)
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Configuracao de quantizacao 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Carregar tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Carregar modelo com quantizacao
print("Carregando modelo base...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

print(f"Modelo {MODEL_NAME} carregado com sucesso!")
print(f"Parametros totais: {model.num_parameters():,}")

---
## 5. Configuracao do LoRA

LoRA (Low-Rank Adaptation) permite treinar apenas uma fracao dos parametros do modelo, tornando o fine-tuning muito mais eficiente.

In [ ]:
# Configuracao LoRA
lora_config = LoraConfig(
    r=16,                          # Rank de baixa dimensao
    lora_alpha=32,                 # Escala dos pesos LoRA
    target_modules=[               # Modulos alvo para adaptacao
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print("LoRA configurado! O SFTTrainer aplicara automaticamente.")

---
## 6. Configuracao do Treinamento

In [ ]:
# Configuracao de treinamento
training_args = TrainingArguments(
    output_dir="../models/assistente_medico_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0
)

print("Configuracao de treinamento definida!")
print(f"Epocas: {training_args.num_train_epochs}")
print(f"Batch size efetivo: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}")

---
## 7. Treinamento do Modelo

O treinamento utiliza o SFTTrainer (Supervised Fine-Tuning Trainer) do Hugging Face TRL.

In [ ]:
# Verificar colunas do dataset
print(f"Colunas disponiveis: {dataset['train'].column_names}")
print(f"\nExemplo do dataset:")
print(dataset['train'][0])

In [ ]:
# Configurar o trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    peft_config=lora_config
)

print("Trainer configurado!")
print(f"Iniciando treinamento com {len(dataset['train'])} exemplos...")

In [ ]:
# Iniciar treinamento
# IMPORTANTE: Em um ambiente real, este processo pode levar varias horas
# Para demo, podemos executar poucas epocas ou usar dados reduzidos

print("="*50)
print("INICIANDO TREINAMENTO")
print("="*50)

try:
    train_result = trainer.train()
    
    print("\n" + "="*50)
    print("TREINAMENTO CONCLUIDO!")
    print("="*50)
    
    # Metricas do treinamento
    metrics = train_result.metrics
    print(f"\nMetricas:")
    print(f"  Loss final: {metrics['train_loss']:.4f}")
    print(f"  Tempo total: {metrics['train_runtime']:.0f} segundos")
    
except Exception as e:
    print(f"Erro durante treinamento: {e}")
    print("Continuando com demonstracao...")

---
## 8. Avaliacao do Modelo

In [ ]:
# Avaliacao do modelo
print("Avaliando modelo...")

eval_results = trainer.evaluate()
print(f"\nResultados da avaliacao:")
print(f"  Eval Loss: {eval_results['eval_loss']:.4f}")

# Perplexidade
import math
perplexity = math.exp(eval_results['eval_loss'])
print(f"  Perplexidade: {perplexity:.2f}")

---
## 9. Teste do Modelo Fine-Tuned

In [ ]:
# Funcao para gerar respostas
def gerar_resposta(prompt_text, max_new_tokens=256):
    """Gera resposta usando o modelo fine-tuned."""
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    resposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return resposta

# Testes
testes = [
    "### Instruction:\nVoce e um assistente medico especializado no hospital. Responda sobre Pneumonia Hospitalar.\n\n### Input:\nQual o protocolo para Pneumonia Hospitalar?\n\n### Response:\n",
    "### Instruction:\nVoce e um assistente medico especializado em Terapia Intensiva.\n\n### Input:\nQuando devemos suspeitar de sepse em um paciente internado?\n\n### Response:\n"
]

for i, teste in enumerate(testes):
    print(f"\n{'='*50}")
    print(f"TESTE {i+1}")
    print(f"{'='*50}")
    resposta = gerar_resposta(teste)
    # Extrair apenas a resposta (apos ### Response:)
    if "### Response:" in resposta:
        resposta_final = resposta.split("### Response:")[1].strip()
    else:
        resposta_final = resposta
    print(f"Resposta:\n{resposta_final}")

---
## 10. Salvamento e Exportacao do Modelo

In [ ]:
# Criar diretorio de saida
output_dir = "../models/assistente_medico_final"
os.makedirs(output_dir, exist_ok=True)

# Salvar modelo LoRA
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Modelo salvo em: {output_dir}")

# Salvar metadados do treinamento
metadados_treinamento = {
    "modelo_base": MODEL_NAME,
    "tecnica": "QLoRA",
    "lora_rank": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "epochs": training_args.num_train_epochs,
    "learning_rate": training_args.learning_rate,
    "batch_size": training_args.per_device_train_batch_size,
    "total_exemplos": len(dataset['train']),
    "eval_loss": eval_results['eval_loss'],
    "perplexidade": perplexity
}

with open(os.path.join(output_dir, "metadados_treinamento.json"), 'w') as f:
    json.dump(metadados_treinamento, f, indent=2)

print(f"\nMetadados salvos:")
print(json.dumps(metadados_treinamento, indent=2))

---
## 11. Resumo

### O que foi feito neste notebook:
1. Configuracao do modelo base com quantizacao 4-bit (QLoRA)
2. Aplicacao de LoRA para treinamento eficiente
3. Treinamento do modelo com dados medicos
4. Avaliacao do modelo (loss, perplexidade)
5. Teste com perguntas medicas
6. Salvamento do modelo fine-tuned

### Metricas importantes:
- **Loss**: Erro do modelo (menor e melhor)
- **Perplexidade**: Incerteza do modelo (menor e melhor)
- **Parametros treinaveis**: Apenas ~1% do total (eficiencia do LoRA)

### Proximo notebook:
O notebook `03_langchain_fundamentos.ipynb` apresentara os conceitos do LangChain para integrar o modelo com chains, prompts e agents.

In [ ]:
print("\n=== NOTEBOOK 02 CONCLUIDO ===")
print("Modelo fine-tuned salvo e pronto para uso!")